<a href="https://colab.research.google.com/github/harshit00052/super-store-data-analysis/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import plotly.express as px
from babel.messages import catalog
from plotly.graph_objs.histogram.marker.colorbar import title
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [6]:
df = pd.read_csv('ecommerce_sales_analytics_5000.csv')

## Understand the dataset

In [7]:
df.sample(1)

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
2689,12690,5/13/2029,1837,Clothing,West,3,473.59,0.01,Wallet,6,3.7,1406.56


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          5000 non-null   int64  
 1   order_date        5000 non-null   object 
 2   customer_id       5000 non-null   int64  
 3   product_category  5000 non-null   object 
 4   region            5000 non-null   object 
 5   quantity          5000 non-null   int64  
 6   unit_price        5000 non-null   float64
 7   discount          5000 non-null   float64
 8   payment_method    5000 non-null   object 
 9   delivery_days     5000 non-null   int64  
 10  customer_rating   5000 non-null   float64
 11  revenue           5000 non-null   float64
dtypes: float64(4), int64(4), object(4)
memory usage: 468.9+ KB


## Check data quality

In [9]:
df[df.duplicated()]  # 0 row
# df[df['order_id'].duplicated()] # 0 row

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue


In [10]:
df['order_date'] = pd.to_datetime(df['order_date'])

## Feature Engineering

In [11]:
df['month'] = df['order_date'].dt.month_name()

In [12]:
df['month_num'] = df['order_date'].dt.month

In [13]:
df['year_quarter'] = df['order_date'].dt.quarter

In [14]:
df['year'] = df['order_date'].dt.year

In [15]:
df = df[df['year'] != 2035]

## Clean text columns and Handle outliers

In [16]:
df['product_category'] = df['product_category'].astype(str).str.strip()
df['region'] = df['region'].astype(str).str.strip()
df['payment_method'] = df['payment_method'].astype(str).str.strip()

In [17]:
df = df[df['quantity']>0]

In [18]:
df = df[df['delivery_days'] > 0]

In [19]:
df.info()
# df.sample(2)

<class 'pandas.core.frame.DataFrame'>
Index: 4748 entries, 0 to 4747
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          4748 non-null   int64         
 1   order_date        4748 non-null   datetime64[ns]
 2   customer_id       4748 non-null   int64         
 3   product_category  4748 non-null   object        
 4   region            4748 non-null   object        
 5   quantity          4748 non-null   int64         
 6   unit_price        4748 non-null   float64       
 7   discount          4748 non-null   float64       
 8   payment_method    4748 non-null   object        
 9   delivery_days     4748 non-null   int64         
 10  customer_rating   4748 non-null   float64       
 11  revenue           4748 non-null   float64       
 12  month             4748 non-null   object        
 13  month_num         4748 non-null   int32         
 14  year_quarter      4748 non-nu

## Group the data for data analysis and visualization

In [20]:
yearlyAnalysis = df.groupby('year').agg({'revenue':'sum', 'quantity':'sum'}).sort_index(ascending=False)
yearlyAnalysis = yearlyAnalysis.reset_index()

In [21]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True)
fig.add_trace(go.Scatter(x=yearlyAnalysis['year'], y=yearlyAnalysis['revenue'], name="Revenue"),row=1,col=1)
fig.add_trace(go.Scatter(x=yearlyAnalysis['year'], y=yearlyAnalysis['quantity'], name="Quantity"),row=2,col=1)
fig.update_layout(height = 600, title="Year Wise Revenue and Quantity")
fig.show()

Revenue and quantity move in a similar direction across most years, indicating a positive relationship between units sold and total revenue.
2033 records the highest revenue as well as one of the highest quantities sold.
2034 shows a noticeable decline in both revenue and quantity compared to the previous year.

#### Recommendation
Let's Analyze which product categories contributed most to the decline and strengthen promotions for those categories.

In [22]:
temp_df = df[df['year'] == 2034]

In [23]:
last_year_ds = temp_df.groupby(['month', 'month_num']).agg({'revenue':'sum', 'quantity':'sum'}).reset_index().sort_values(by=['month_num'])

In [24]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True)
fig.add_trace(go.Scatter(x=last_year_ds['month'], y=last_year_ds['revenue'], name="Revenue"),row=1,col=1)
fig.add_trace(go.Scatter(x=last_year_ds['month'], y=last_year_ds['quantity'], name="Quantity"),row=2,col=1)
fig.update_layout(height=600, title="Monthly Revenue and Quantity for 2034")

#### From above graphs we can say only july month have the highest sale and rest month the performance is very bad
##### let's find which product category performance have affected the sales

In [25]:
req_year_ds = temp_df.groupby(['product_category','month', 'month_num'])['order_id'].count().reset_index().sort_values(by='month_num')

In [26]:
fig = go.Figure()

for category in req_year_ds['product_category'].unique():
    temp = req_year_ds[req_year_ds['product_category'] == category]

    fig.add_trace(go.Scatter(y=temp['order_id'], x=temp['month'], name=category))

fig.update_layout(title='monthly sale of each category', xaxis_title='month', yaxis_title='Number of orders')
fig.show()

#### from above graph we can clearly see the reason for 2034 poor performance are beauty products and home appliances
#### Recommendation
strengthen promotions for these two category and focus on the user reviews for these products


In [27]:
monthly_analysis = df.groupby(['month', 'month_num']).agg({'revenue':'sum', 'quantity':'sum'}).reset_index().sort_values('month_num')
monthly_analysis.head(3)

,month,month_num,revenue,quantity
4,January,1,393564.88,1634
3,February,2,384742.47,1458
7,March,3,392913.79,1612


In [28]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['Month wise revenue', 'month wise quantity'])
fig.add_trace(go.Scatter(x=monthly_analysis['month'], y=monthly_analysis['revenue'], name="Revenue"),row=1,col=1)
fig.add_trace(go.Scatter(x=monthly_analysis['month'], y=monthly_analysis['quantity'], name="Quantity"),row=2,col=1)
fig.update_layout(height=600, title="Monthly Revenue and Quantity")
fig.show()

#### I can see in the May month we sold most and in starting of the year we see very up's and down's which effects or revenue and rest of the month is almost content
#### I will advise you to keep most demanding product category of product in the starting month's and focus of advertisement

In [29]:
df.groupby('year_quarter').agg({'revenue':'sum', 'quantity':'sum'}).sort_values(by=['revenue'], ascending=False)

,revenue,quantity
year_quarter,,
3,1245562.16,4831
2,1235041.92,4866
4,1219217.16,4810
1,1171221.14,4704


In [30]:
df.groupby('customer_id').agg({'order_id':'count', 'revenue':'sum'}).reset_index().sort_values(by=['order_id'], ascending=False)

,customer_id,order_id,revenue
38,1038,12,10483.21
912,1928,12,9480.66
659,1669,12,10572.84
653,1663,12,14637.83
263,1267,11,12447.79
...,...,...,...
181,1184,1,522.12
185,1188,1,1438.50
668,1679,1,1568.46
5,1005,1,1064.95


In [31]:
df.groupby('product_category').agg({'revenue':'sum'}).sort_values(by=['revenue'], ascending=False)

,revenue
product_category,
Electronics,1748795.42
Clothing,1447304.19
Home,937269.33
Beauty,737673.44


In [32]:
df.pivot_table(index=['year','product_category'], columns='year_quarter', values='revenue', aggfunc=sum).reset_index().sample(5)

/tmp/ipykernel_1197/509024725.py:1: FutureWarning:

The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.



year_quarter,year,product_category,1,2,3,4
39,2031,Home,18798.83,21990.84,13483.37,13980.64
8,2024,Beauty,13034.58,17159.23,11761.77,11409.37
32,2030,Beauty,9364.73,12675.95,15671.72,18008.93
14,2025,Electronics,30351.53,31383.43,38946.70,32560.05
26,2028,Electronics,43624.82,45539.38,27346.23,29782.64


In [33]:
df.pivot_table(index=['year', 'product_category'], columns='region', values='revenue', aggfunc=sum).reset_index().sample(5)

/tmp/ipykernel_1197/2996077394.py:1: FutureWarning:

The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.



region,year,product_category,East,North,South,West
43,2032,Home,19720.48,13628.27,22176.65,15617.70
16,2026,Beauty,13403.88,14640.68,12856.00,7379.84
26,2028,Electronics,21859.87,40904.74,37912.23,45616.23
15,2025,Home,20268.64,11470.74,19021.17,23753.79
9,2024,Clothing,17663.35,22821.60,26795.07,35038.78


In [34]:
df.pivot_table(index=['year', 'product_category'], columns='region', values='order_id', aggfunc='count').reset_index().sample(5)

region,year,product_category,East,North,South,West
10,2024,Electronics,38,32,35,24
41,2032,Clothing,25,38,29,31
40,2032,Beauty,13,13,17,15
21,2027,Clothing,25,26,30,27
31,2029,Home,17,15,16,13


In [35]:
df.pivot_table(index=['product_category'], columns='year', values='order_id', aggfunc='count').reset_index()

# Customers consistently prefer Electronics over other categories :- Increase inventory allocation for Electronics and introduce premium product variants
# Beauty contributes the least to total unit sales :- Investigate pricing , Consider bundling Beauty products with high-selling categories.
# Sales fluctuate from year to year :- month by month analysis may help here to find sale trend


year,product_category,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
0,Beauty,53,49,49,46,46,55,54,62,55,48,58,62,57
1,Clothing,107,108,106,105,125,108,108,112,100,114,123,102,120
2,Electronics,130,129,129,133,126,126,141,130,134,134,115,142,124
3,Home,75,79,82,81,68,76,63,61,76,69,70,59,64


In [36]:
df.groupby('product_category').agg({'customer_rating':'mean', 'delivery_days':'mean'}).round(2)

# customer rating is very low specially for electronic and home :- Conduct customer feedback analysis and review product quality, delivery experience, and after-sales support to identify the root causes of dissatisfaction.

# delivery days are average but can be reduced

,customer_rating,delivery_days
product_category,,
Beauty,3.00,6.16
Clothing,3.02,6.15
Electronics,2.95,6.08
Home,2.93,6.12


In [37]:
df.groupby('payment_method')['order_id'].count()

,order_id
payment_method,
COD,1690
Card,2150
Wallet,908
